# Wadjet v2 — Landmark Classifier Training (ONNX Export)

**Purpose**: Train an EfficientNetV2-B0 classifier for 52 Egyptian landmarks, export as ONNX for browser + server.

## CRITICAL RULES
1. **float32 ONLY** — NO `mixed_float16` (causes fused ops that crash in any runtime)
2. Export via **`tf2onnx.convert.from_keras()`** → `.onnx` file
3. Quantize via **`onnxruntime.quantization.quantize_dynamic()`** → uint8
4. Load in browser with **`ort.InferenceSession.create()`** (ONNX Runtime Web)
5. Architecture: **EfficientNetV2-B0** at **224×224**

## Dataset
- Source: `naderelakany/wadjet-tfrecords` on Kaggle
- 52 classes (Egyptian landmarks), ~46K train / 4.5K val / 4.5K test
- TFRecords at 384×384 JPEG, resized to 224×224 during training
- Train includes augmented images (21K originals + 25K augmented)

## Training Strategy
- **Phase 1**: Head only (5 epochs, backbone frozen, lr=1e-3)
- **Phase 2**: Fine-tune top 70% (50 epochs, lr=1e-4 with warmup + cosine decay)
- Categorical Focal Loss (γ=2.0) for MixUp/CutMix compatibility
- MixUp (α=0.2) + CutMix (α=1.0) batch-level augmentation

In [ ]:
# ============================================================
# Configuration
# ============================================================
import os

# Data paths (Kaggle mount — landmark TFRecords at root level)
DATA_DIR = "/kaggle/input/wadjet-tfrecords"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")

# Model
IMAGE_SIZE = 224
NUM_CLASSES = 52

# Training
BATCH_SIZE = 16
SEED = 42
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 50
HEAD_LR = 1e-3
FINETUNE_LR = 1e-4
FINETUNE_MIN_LR = 5e-6
WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
FOCAL_GAMMA = 2.0
UNFREEZE_RATIO = 0.7
PATIENCE = 15
SHUFFLE_BUFFER = 1024

# Augmentation
USE_MIXUP = True
USE_CUTMIX = True
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0
MIXUP_PROB = 0.5     # probability of MixUp vs CutMix per batch

# Warmup
WARMUP_EPOCHS_P1 = 1
WARMUP_EPOCHS_P2 = 3

# Output
OUTPUT_DIR = "/kaggle/working"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_landmark_model.keras")
ONNX_FLOAT_PATH = os.path.join(OUTPUT_DIR, "landmark_classifier.onnx")
ONNX_UINT8_PATH = os.path.join(OUTPUT_DIR, "landmark_classifier_uint8.onnx")
METADATA_PATH = os.path.join(OUTPUT_DIR, "model_metadata.json")

# WandB (set True if configured)
USE_WANDB = False
WANDB_PROJECT = "wadjet-landmark-v2"

In [ ]:
# ============================================================
# Install & Import
# ============================================================
!pip install tf2onnx onnxruntime -q

import glob
import json
import math
import numpy as np
import tensorflow as tf
from pathlib import Path

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ============================================================
# GPU + float32 Verification
# ============================================================
print(f"TensorFlow version: {tf.__version__}")

# CRITICAL: Verify NO mixed precision
policy = tf.keras.mixed_precision.global_policy()
print(f"Global dtype policy: {policy}")
assert 'float16' not in str(policy), "ABORT: mixed_float16 detected! Must use float32."

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

In [ ]:
# ============================================================
# TFRecord Parsing
# ============================================================
FEATURE_DESC = {
    "image_raw": tf.io.FixedLenFeature([], tf.string),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse_example(serialized):
    """Parse TFRecord — keep raw JPEG bytes, decode later."""
    return tf.io.parse_single_example(serialized, FEATURE_DESC)

def preprocess_train_onehot(features):
    """Decode + one-hot labels (needed for MixUp/CutMix)."""
    image = tf.io.decode_jpeg(features["image_raw"], channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(tf.cast(features["label"], tf.int32), NUM_CLASSES)
    return image, label

def preprocess_eval_onehot(features):
    """Decode + one-hot labels for validation (MUST match CategoricalFocalLoss)."""
    image = tf.io.decode_jpeg(features["image_raw"], channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(tf.cast(features["label"], tf.int32), NUM_CLASSES)
    return image, label

def preprocess_eval_sparse(features):
    """Decode with sparse integer labels (for final test evaluation / sklearn metrics)."""
    image = tf.io.decode_jpeg(features["image_raw"], channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, features["label"]

In [ ]:
# ============================================================
# Dataset Statistics
# ============================================================
print("Counting samples per split...")
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.tfrecord")))
val_files = sorted(glob.glob(os.path.join(VAL_DIR, "*.tfrecord")))
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.tfrecord")))

print(f"TFRecord files: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

train_count = sum(1 for _ in tf.data.TFRecordDataset(train_files))
val_count = sum(1 for _ in tf.data.TFRecordDataset(val_files))
test_count = sum(1 for _ in tf.data.TFRecordDataset(test_files))

print(f"Train: {train_count:,} | Val: {val_count:,} | Test: {test_count:,}")
print(f"Total: {train_count + val_count + test_count:,}")

In [ ]:
# ============================================================
# Augmentation (flip OK for landmarks, plus MixUp/CutMix)
# ============================================================
def rotate_image(image, angle):
    """Rotate image by angle (radians) around center."""
    cos_a = tf.cos(angle)
    sin_a = tf.sin(angle)
    h = tf.cast(tf.shape(image)[0], tf.float32)
    w = tf.cast(tf.shape(image)[1], tf.float32)
    cx, cy = w / 2.0, h / 2.0
    transform = [
        cos_a, -sin_a, cx - cx * cos_a + cy * sin_a,
        sin_a,  cos_a, cy - cx * sin_a - cy * cos_a,
        0.0, 0.0
    ]
    transformed = tf.raw_ops.ImageProjectiveTransformV3(
        images=tf.expand_dims(image, 0),
        transforms=tf.reshape(transform, [1, 8]),
        output_shape=tf.shape(image)[:2],
        interpolation="BILINEAR",
        fill_mode="REFLECT",
        fill_value=0.0
    )
    return transformed[0]

def random_erasing(image, probability=0.2, area_range=(0.02, 0.33)):
    """Random erasing (cutout) on a single image."""
    if tf.random.uniform([]) > probability:
        return image
    h = tf.shape(image)[0]
    w = tf.shape(image)[1]
    area = tf.cast(h * w, tf.float32)
    target_area = tf.random.uniform([], area_range[0], area_range[1]) * area
    aspect = tf.random.uniform([], 0.3, 3.3)
    eh = tf.cast(tf.math.sqrt(target_area * aspect), tf.int32)
    ew = tf.cast(tf.math.sqrt(target_area / aspect), tf.int32)
    eh = tf.minimum(eh, h)
    ew = tf.minimum(ew, w)
    y0 = tf.random.uniform([], 0, h - eh + 1, dtype=tf.int32)
    x0 = tf.random.uniform([], 0, w - ew + 1, dtype=tf.int32)
    mask = tf.ones_like(image)
    padding = tf.zeros([eh, ew, 3])
    indices_y = tf.range(y0, y0 + eh)
    indices_x = tf.range(x0, x0 + ew)
    # Simple approach: set region to mean color
    mean_color = tf.reduce_mean(image)
    patch = tf.fill([eh, ew, 3], mean_color)
    image = tf.tensor_scatter_nd_update(
        image,
        tf.reshape(tf.stack(tf.meshgrid(indices_y, indices_x, indexing='ij'), axis=-1), [-1, 2]),
        tf.reshape(patch, [-1, 3])
    )
    return image

@tf.function
def augment_image(image, label):
    """Per-image augmentation (landmarks: flip OK)."""
    image = tf.image.random_flip_left_right(image)
    angle = tf.random.uniform([], -18.0, 18.0) * (3.14159265 / 180.0)
    image = rotate_image(image, angle)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.02)
    image = random_erasing(image, probability=0.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

In [ ]:
# ============================================================
# MixUp + CutMix (batch-level augmentation)
# ============================================================
def sample_beta(alpha):
    """Sample from Beta(alpha, alpha) using gamma distribution.
    Beta(a,b) = Gamma(a) / (Gamma(a) + Gamma(b)) where a=b=alpha.
    """
    g1 = tf.random.gamma([1], alpha=alpha)
    g2 = tf.random.gamma([1], alpha=alpha)
    return g1 / (g1 + g2 + 1e-7)

@tf.function
def mixup_batch(images, labels):
    """MixUp: linearly interpolate random pairs."""
    batch_size = tf.shape(images)[0]
    # Sample lambda from Beta(alpha, alpha), ensure lambda >= 0.5
    lam = sample_beta(MIXUP_ALPHA)[0]
    lam = tf.maximum(lam, 1.0 - lam)  # keep dominant sample
    indices = tf.random.shuffle(tf.range(batch_size))
    images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return images, labels

@tf.function
def cutmix_batch(images, labels):
    """CutMix: paste rectangular region from random partner."""
    batch_size = tf.shape(images)[0]
    h = tf.cast(tf.shape(images)[1], tf.float32)
    w = tf.cast(tf.shape(images)[2], tf.float32)

    lam = sample_beta(CUTMIX_ALPHA)[0]
    cut_ratio = tf.sqrt(1.0 - lam)
    cut_h = tf.cast(h * cut_ratio, tf.int32)
    cut_w = tf.cast(w * cut_ratio, tf.int32)

    cy = tf.random.uniform([], 0, tf.cast(h, tf.int32), dtype=tf.int32)
    cx = tf.random.uniform([], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    y1 = tf.maximum(cy - cut_h // 2, 0)
    y2 = tf.minimum(cy + cut_h // 2, tf.cast(h, tf.int32))
    x1 = tf.maximum(cx - cut_w // 2, 0)
    x2 = tf.minimum(cx + cut_w // 2, tf.cast(w, tf.int32))

    indices = tf.random.shuffle(tf.range(batch_size))
    shuffled_images = tf.gather(images, indices)
    shuffled_labels = tf.gather(labels, indices)

    # Create mask: 1 for cut region, 0 for keep
    mask_h = tf.cast(tf.range(tf.cast(h, tf.int32)), tf.int32)
    mask_w = tf.cast(tf.range(tf.cast(w, tf.int32)), tf.int32)
    mask_h = tf.cast((mask_h >= y1) & (mask_h < y2), tf.float32)
    mask_w = tf.cast((mask_w >= x1) & (mask_w < x2), tf.float32)
    mask = tf.einsum('h,w->hw', mask_h, mask_w)
    mask = tf.expand_dims(tf.expand_dims(mask, 0), -1)  # [1, H, W, 1]

    images = images * (1.0 - mask) + shuffled_images * mask

    # Recompute lambda based on actual cut area
    actual_lam = 1.0 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / (h * w)
    labels = actual_lam * labels + (1.0 - actual_lam) * shuffled_labels
    return images, labels

@tf.function
def apply_mixup_or_cutmix(images, labels):
    """Randomly apply MixUp or CutMix per batch."""
    if not (USE_MIXUP or USE_CUTMIX):
        return images, labels
    if USE_MIXUP and USE_CUTMIX:
        return tf.cond(
            tf.random.uniform([]) < MIXUP_PROB,
            lambda: mixup_batch(images, labels),
            lambda: cutmix_batch(images, labels)
        )
    elif USE_MIXUP:
        return mixup_batch(images, labels)
    else:
        return cutmix_batch(images, labels)

In [ ]:
# ============================================================
# Build Datasets
# ============================================================
def build_train_ds():
    """Build training dataset with augmentation + MixUp/CutMix."""
    ds = tf.data.TFRecordDataset(train_files, num_parallel_reads=4)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(SHUFFLE_BUFFER, seed=SEED)
    ds = ds.map(preprocess_train_onehot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(augment_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.map(apply_mixup_or_cutmix, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

def build_eval_ds_onehot(files):
    """Build eval dataset with one-hot labels (for val — must match CategoricalFocalLoss)."""
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=4)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess_eval_onehot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

def build_eval_ds_sparse(files):
    """Build eval dataset with sparse labels (for test — sklearn metrics need integers)."""
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=4)
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(preprocess_eval_sparse, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("Building datasets...")
train_ds = build_train_ds()
val_ds = build_eval_ds_onehot(val_files)    # one-hot — matches CategoricalFocalLoss
test_ds = build_eval_ds_sparse(test_files)  # sparse — for argmax + sklearn metrics

# Sanity check
for images, labels in train_ds.take(1):
    print(f"Train batch — images: {images.shape}, dtype: {images.dtype}")
    print(f"Train batch — labels: {labels.shape} (one-hot), dtype: {labels.dtype}")
    assert images.dtype == tf.float32
    assert labels.shape[-1] == NUM_CLASSES, f"Train labels must be one-hot ({NUM_CLASSES})"
for images, labels in val_ds.take(1):
    print(f"Val batch — images: {images.shape}")
    print(f"Val batch — labels: {labels.shape} (one-hot), dtype: {labels.dtype}")
    assert labels.shape[-1] == NUM_CLASSES, f"Val labels must be one-hot to match CategoricalFocalLoss"
for images, labels in test_ds.take(1):
    print(f"Test batch — images: {images.shape}")
    print(f"Test batch — labels: {labels.shape} (sparse int), dtype: {labels.dtype}")
print("All sanity checks passed.")

In [ ]:
# ============================================================
# Loss Functions
# ============================================================
import keras

@keras.saving.register_keras_serializable()
class CategoricalFocalLoss(tf.keras.losses.Loss):
    """Focal loss for one-hot / soft labels (MixUp/CutMix compatible)."""
    def __init__(self, gamma=2.0, class_weights=None, name="categorical_focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self._class_weights_list = class_weights

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        y_true = tf.cast(y_true, tf.float32)
        focal_weight = tf.pow(1.0 - y_pred, self.gamma)
        ce = -tf.math.log(y_pred)
        if self._class_weights_list is not None:
            cw = tf.constant(self._class_weights_list, dtype=tf.float32)
            ce = ce * cw  # broadcasts [num_classes] -> [batch, num_classes]
        return tf.reduce_mean(tf.reduce_sum(y_true * focal_weight * ce, axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "class_weights": self._class_weights_list})
        return config

@keras.saving.register_keras_serializable()
class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    """Focal loss for sparse integer labels (evaluation)."""
    def __init__(self, gamma=2.0, class_weights=None, name="sparse_focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self._class_weights_list = class_weights

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        batch_indices = tf.range(tf.shape(y_true)[0])
        indices = tf.stack([batch_indices, y_true], axis=1)
        p_t = tf.gather_nd(y_pred, indices)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        ce = -tf.math.log(p_t)
        if self._class_weights_list is not None:
            cw = tf.constant(self._class_weights_list, dtype=tf.float32)
            ce = ce * tf.gather(cw, y_true)
        return tf.reduce_mean(focal_weight * ce)

    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "class_weights": self._class_weights_list})
        return config

In [ ]:
# ============================================================
# Class Weights (inverse frequency, clipped)
# ============================================================
print("Computing class weights from training data...")
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
label_ds = (
    tf.data.TFRecordDataset(train_files)
    .map(lambda x: tf.io.parse_single_example(
        x, {"label": tf.io.FixedLenFeature([], tf.int64)})["label"],
        num_parallel_calls=4)
    .batch(1024)
)
for batch_labels in label_ds:
    for lbl in batch_labels.numpy():
        class_counts[int(lbl)] += 1

total_samples = class_counts.sum()
raw_weights = total_samples / (NUM_CLASSES * class_counts.astype(np.float64))
CLASS_WEIGHTS_LIST = np.clip(raw_weights, 0.5, 3.0).tolist()
CLASS_WEIGHTS_DICT = {i: w for i, w in enumerate(CLASS_WEIGHTS_LIST)}

print(f"Total train samples: {total_samples:,}")
print(f"Weight range: [{min(CLASS_WEIGHTS_LIST):.2f}, {max(CLASS_WEIGHTS_LIST):.2f}]")
print(f"Class counts range: [{class_counts.min()}, {class_counts.max()}]")

In [ ]:
# ============================================================
# LR Schedule: Warmup + Cosine Decay
# ============================================================
@keras.saving.register_keras_serializable()
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, min_lr, total_steps, warmup_steps, **kwargs):
        super().__init__(**kwargs)
        self.initial_lr = initial_lr
        self.min_lr = min_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0))
        decay_steps = tf.cast(self.total_steps - self.warmup_steps, tf.float32)
        progress = (step - tf.cast(self.warmup_steps, tf.float32)) / tf.maximum(decay_steps, 1.0)
        cosine_decay = 0.5 * (1.0 + tf.cos(3.14159265 * progress))
        cosine_lr = self.min_lr + (self.initial_lr - self.min_lr) * cosine_decay
        return tf.where(step < tf.cast(self.warmup_steps, tf.float32), warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "initial_lr": self.initial_lr, "min_lr": self.min_lr,
            "total_steps": self.total_steps, "warmup_steps": self.warmup_steps,
        }

In [ ]:
# ============================================================
# Model Architecture: EfficientNetV2-B0
# ============================================================
def build_model(num_classes, image_size, dropout=DROPOUT, freeze_base=True):
    """Build EfficientNetV2-B0 with classification head."""
    base = tf.keras.applications.EfficientNetV2B0(
        input_shape=(image_size, image_size, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(image_size, image_size, 3), name="image_input")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = tf.keras.layers.Dropout(dropout, name="head_dropout")(x)
    outputs = tf.keras.layers.Dense(
        num_classes, activation="softmax", dtype="float32", name="predictions"
    )(x)

    model = tf.keras.Model(inputs, outputs, name="wadjet_landmark_classifier")
    return model, base

model, backbone = build_model(NUM_CLASSES, IMAGE_SIZE, freeze_base=True)
model.summary(show_trainable=True, expand_nested=False)

In [ ]:
# ============================================================
# Phase 1: Head Training (backbone frozen)
# ============================================================
print("=" * 60)
print("PHASE 1: Training classification head (backbone frozen)")
print("=" * 60)

steps_per_epoch = train_count // BATCH_SIZE + 1
p1_total_steps = steps_per_epoch * HEAD_EPOCHS
p1_warmup_steps = steps_per_epoch * WARMUP_EPOCHS_P1

p1_lr = WarmupCosineDecay(
    initial_lr=HEAD_LR, min_lr=1e-4,
    total_steps=p1_total_steps, warmup_steps=p1_warmup_steps
)

# Both train_ds and val_ds use one-hot labels → CategoricalFocalLoss is correct for both
train_loss = CategoricalFocalLoss(gamma=FOCAL_GAMMA, class_weights=CLASS_WEIGHTS_LIST)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=p1_lr, weight_decay=WEIGHT_DECAY),
    loss=train_loss,
    metrics=["accuracy"],
)

callbacks_p1 = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_loss", mode="min",
        save_best_only=True, verbose=1
    ),
]

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=callbacks_p1,
)

print(f"Phase 1 complete. Best val_loss: {min(history_p1.history['val_loss']):.4f}")

In [ ]:
# ============================================================
# Phase 2: Fine-tuning (top 70% of backbone unfrozen)
# ============================================================
print("=" * 60)
print("PHASE 2: Fine-tuning (unfreezing top 70% of backbone)")
print("=" * 60)

backbone.trainable = True
freeze_until = int(len(backbone.layers) * (1.0 - UNFREEZE_RATIO))
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in backbone.layers if l.trainable)
print(f"Backbone: {trainable_count}/{len(backbone.layers)} layers trainable")

p2_total_steps = steps_per_epoch * FINETUNE_EPOCHS
p2_warmup_steps = steps_per_epoch * WARMUP_EPOCHS_P2

p2_lr = WarmupCosineDecay(
    initial_lr=FINETUNE_LR, min_lr=FINETUNE_MIN_LR,
    total_steps=p2_total_steps, warmup_steps=p2_warmup_steps
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=p2_lr, weight_decay=WEIGHT_DECAY),
    loss=CategoricalFocalLoss(gamma=FOCAL_GAMMA, class_weights=CLASS_WEIGHTS_LIST),
    metrics=["accuracy"],
)

callbacks_p2 = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_loss", mode="min",
        save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE,
        min_delta=0.001, restore_best_weights=True, verbose=1
    ),
]
if USE_WANDB:
    import wandb
    from wandb.integration.keras import WandbMetricsLogger
    wandb.init(project=WANDB_PROJECT, name="phase2-finetune", config={
        "image_size": IMAGE_SIZE, "backbone": "EfficientNetV2B0",
        "phase": 2, "lr": FINETUNE_LR, "epochs": FINETUNE_EPOCHS,
    })
    callbacks_p2.append(WandbMetricsLogger(log_freq="epoch"))

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=callbacks_p2,
)

print(f"Phase 2 complete. Best val_loss: {min(history_p2.history['val_loss']):.4f}")
if USE_WANDB:
    wandb.finish()

In [ ]:
# ============================================================
# Evaluation + Per-Class Metrics
# ============================================================
from sklearn.metrics import classification_report

# Load best model
model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "CategoricalFocalLoss": CategoricalFocalLoss,
        "SparseCategoricalFocalLoss": SparseCategoricalFocalLoss,
        "WarmupCosineDecay": WarmupCosineDecay,
    }
)

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy().astype(int))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = np.mean(y_true == y_pred)
print(f"Test accuracy: {accuracy:.4f}")

# Per-class report
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
class_f1 = {int(k): v['f1-score'] for k, v in report.items() if k.isdigit()}
worst = sorted(class_f1.items(), key=lambda x: x[1])[:10]
print("\nWorst 10 classes by F1-score:")
for idx, f1 in worst:
    print(f"  Class {idx}: F1={f1:.3f}, count={int(class_counts[idx])}")

In [ ]:
# ============================================================
# ONNX Export (replaces TF.js export entirely)
# ============================================================
import tf2onnx

print(f"tf2onnx version: {tf2onnx.__version__}")

# Define input signature — NHWC format matching Keras
spec = (tf.TensorSpec((None, IMAGE_SIZE, IMAGE_SIZE, 3), tf.float32, name="input"),)

# Convert Keras model to ONNX
model_onnx, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=17)

# Save full-precision ONNX model
with open(ONNX_FLOAT_PATH, "wb") as f:
    f.write(model_onnx.SerializeToString())

onnx_size = os.path.getsize(ONNX_FLOAT_PATH)
print(f"\nONNX model exported to: {ONNX_FLOAT_PATH}")
print(f"  Size: {onnx_size / 1024:.1f} KB ({onnx_size / (1024*1024):.1f} MB)")

In [ ]:
# ============================================================
# ONNX uint8 Quantization
# ============================================================
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    ONNX_FLOAT_PATH,
    ONNX_UINT8_PATH,
    weight_type=QuantType.QUInt8,
)

uint8_size = os.path.getsize(ONNX_UINT8_PATH)
float_size = os.path.getsize(ONNX_FLOAT_PATH)
print(f"Full-precision ONNX: {float_size / 1024:.1f} KB ({float_size / (1024*1024):.1f} MB)")
print(f"uint8 ONNX: {uint8_size / 1024:.1f} KB ({uint8_size / (1024*1024):.1f} MB)")
print(f"Compression: {(1 - uint8_size / float_size) * 100:.1f}%")

In [ ]:
# ============================================================
# Validate ONNX + Save Outputs
# ============================================================
import onnxruntime as ort

# Validate both ONNX models
for onnx_path, label in [(ONNX_FLOAT_PATH, "float32"), (ONNX_UINT8_PATH, "uint8")]:
    sess = ort.InferenceSession(onnx_path)
    inp = sess.get_inputs()[0]
    out = sess.get_outputs()[0]
    print(f"\n[{label}] ONNX validation:")
    print(f"  Input: name={inp.name}, shape={inp.shape}, type={inp.type}")
    print(f"  Output: name={out.name}, shape={out.shape}, type={out.type}")

    dummy = np.random.rand(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
    result = sess.run(None, {inp.name: dummy})
    probs = result[0]
    print(f"  Output shape: {probs.shape}, sum: {probs.sum():.4f}")
    assert probs.shape[-1] == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, got {probs.shape[-1]}"
    print(f"  ✅ PASS")

# Cross-validate Keras vs ONNX float32
dummy = np.random.rand(1, IMAGE_SIZE, IMAGE_SIZE, 3).astype(np.float32)
keras_out = model.predict(dummy, verbose=0)
onnx_sess = ort.InferenceSession(ONNX_FLOAT_PATH)
onnx_out = onnx_sess.run(None, {onnx_sess.get_inputs()[0].name: dummy})[0]
max_diff = np.abs(keras_out - onnx_out).max()
print(f"\nKeras vs ONNX float32 max diff: {max_diff:.6f}")
assert max_diff < 1e-4, f"ONNX output diverges from Keras: max_diff={max_diff}"
print("✅ Keras and ONNX outputs match")

# Load landmark class names if available
landmark_meta_path = os.path.join(DATA_DIR, "tfrecord_log.json")
if os.path.exists(landmark_meta_path):
    with open(landmark_meta_path) as f:
        landmark_meta = json.load(f)
    print(f"Loaded landmark metadata")
else:
    landmark_meta = {}

# Save output metadata
output_metadata = {
    "model_name": "wadjet_landmark_classifier_v2",
    "architecture": "EfficientNetV2B0",
    "input_size": IMAGE_SIZE,
    "num_classes": NUM_CLASSES,
    "format": "onnx",
    "load_method": "ort.InferenceSession.create(url)",
    "preprocessing": {
        "normalize": "divide_by_255",
        "input_range": [0.0, 1.0],
    },
    "training": {
        "precision": "float32",
        "loss": f"CategoricalFocalLoss(gamma={FOCAL_GAMMA})",
        "head_epochs": HEAD_EPOCHS,
        "finetune_epochs": FINETUNE_EPOCHS,
        "test_accuracy": float(accuracy),
        "augmentation": ["flip", "rotation", "brightness", "contrast", "saturation", "hue", "erasing", "MixUp", "CutMix"],
    },
}

with open(METADATA_PATH, "w") as f:
    json.dump(output_metadata, f, indent=2)
print(f"Metadata saved to: {METADATA_PATH}")

model.save(BEST_MODEL_PATH)

print("\n" + "=" * 60)
print("ALL OUTPUTS:")
print("=" * 60)
print(f"  1. Keras model (backup): {BEST_MODEL_PATH}")
print(f"  2. ONNX float32: {ONNX_FLOAT_PATH} ({os.path.getsize(ONNX_FLOAT_PATH)/(1024*1024):.1f} MB)")
print(f"  3. ONNX uint8: {ONNX_UINT8_PATH} ({os.path.getsize(ONNX_UINT8_PATH)/(1024*1024):.1f} MB)")
print(f"  4. Metadata: {METADATA_PATH}")

## Download Instructions

After training completes on Kaggle:

1. **Download `landmark_classifier_uint8.onnx`** (recommended for production — single file!)

2. **Download `model_metadata.json`**

3. **Copy to Wadjet v2 project**:
   ```
   models/landmark/onnx/
   ├── landmark_classifier_uint8.onnx
   └── model_metadata.json
   ```

4. **Update `explore.html`**:
   - Replace `tf.loadLayersModel()` with `ort.InferenceSession.create('/models/landmark/onnx/landmark_classifier_uint8.onnx')`
   - Update preprocessing: resize to 224×224 → /255.0 → NHWC format → `new ort.Tensor('float32', data, [1,224,224,3])`
   - Add ONNX Runtime Web CDN (or it may already be loaded globally)

5. **Remove TF.js lazy-load** from `explore.html`

6. **Bump service worker cache version**

7. **Key difference from hieroglyph**: This model uses **224×224** input, not 128×128